#  UN Regions + World Bank Inflation Data Scraper

This notebook scrapes and merges two publicly available datasets:

| Data | Source |
|------|--------|
| **UN Regions (M49 Standard)** | [lukes/ISO-3166 on GitHub](https://github.com/lukes/ISO-3166-Countries-with-Regional-Codes) |
| **Inflation / CPI (Annual %)** | [World Bank Open API](https://data.worldbank.org/indicator/FP.CPI.TOTL.ZG) |

**Output:** A merged CSV file `inflation_by_un_region.csv` ready for analysis.

---

##  Step 1 — Install & Import Libraries

In [ ]:
# Install required libraries (already available in Colab, but included for safety)
!pip install requests beautifulsoup4 pandas --quiet

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
from io import StringIO

print("Libraries loaded successfully.")

---
##  Step 2 — Scrape UN M49 Region Data

We fetch the UN M49 country-to-region mapping from a well-maintained open dataset on GitHub.
It maps every ISO country code to its UN **region** (e.g. *Asia*) and **sub-region** (e.g. *Southern Asia*).

In [ ]:
def scrape_un_regions():
    """
    Fetches the UN M49 country-to-region mapping.
    Source: https://github.com/lukes/ISO-3166-Countries-with-Regional-Codes
    Returns a DataFrame with: country_name, iso3_code, un_region, un_sub_region
    """
    url = (
        "https://raw.githubusercontent.com/lukes/"
        "ISO-3166-Countries-with-Regional-Codes/master/all/all.csv"
    )

    print(" Fetching UN M49 region data ...")
    response = requests.get(url, timeout=15)
    response.raise_for_status()

    df = pd.read_csv(StringIO(response.text))

    # Keep only relevant columns and rename
    df = df[["name", "alpha-3", "region", "sub-region"]].copy()
    df.columns = ["country_name", "iso3_code", "un_region", "un_sub_region"]

    # Drop rows with no region assignment (e.g. Antarctica)
    df = df.dropna(subset=["un_region"])
    df["iso3_code"] = df["iso3_code"].str.upper()

    print(f" {len(df)} countries with UN region data retrieved.")
    return df


un_regions_df = scrape_un_regions()
un_regions_df.head(10)

---
##  Step 3 — Fetch Inflation Data from World Bank API

We call the **World Bank Open API** to retrieve annual inflation (CPI % change) for all countries.

- **Indicator:** `FP.CPI.TOTL.ZG` — Inflation, consumer prices (annual %)
- **Year range:** 2010 – 2023 *(adjustable below)*

In [ ]:
def fetch_inflation_data(start_year=2010, end_year=2023):
    """
    Fetches annual CPI inflation (%) for all countries via World Bank API.
    Indicator: FP.CPI.TOTL.ZG
    Returns a DataFrame with: iso3_code, country_name_wb, year, inflation_pct
    """
    base_url = (
        "https://api.worldbank.org/v2/country/all/indicator/FP.CPI.TOTL.ZG"
        f"?format=json&per_page=10000&date={start_year}:{end_year}"
    )

    all_records = []
    page = 1

    print(f" Fetching World Bank inflation data ({start_year}–{end_year}) ...")

    while True:
        response = requests.get(f"{base_url}&page={page}", timeout=20)
        response.raise_for_status()
        data = response.json()

        if len(data) < 2 or not data[1]:
            break

        metadata, records = data[0], data[1]

        for record in records:
            if record.get("value") is not None:
                all_records.append({
                    "iso3_code"       : record["countryiso3code"],
                    "country_name_wb" : record["country"]["value"],
                    "year"            : int(record["date"]),
                    "inflation_pct"   : round(float(record["value"]), 4)
                })

        total_pages = metadata.get("pages", 1)
        print(f"   Page {page}/{total_pages} — {len(all_records)} records so far ...")

        if page >= total_pages:
            break

        page += 1
        time.sleep(0.3)   # polite delay

    df = pd.DataFrame(all_records)
    print(f"\n {len(df)} inflation data points retrieved.")
    return df


inflation_df = fetch_inflation_data(start_year=2010, end_year=2023)
inflation_df.head(10)

---
##  Step 4 — Merge Datasets

In [ ]:
def merge_datasets(un_df, inflation_df):
    """
    Merges UN region data with World Bank inflation data on ISO3 country code.
    """
    merged = pd.merge(
        inflation_df,
        un_df[["iso3_code", "un_region", "un_sub_region"]],
        on="iso3_code",
        how="left"
    )

    # Reorder columns
    merged = merged[[
        "iso3_code",
        "country_name_wb",
        "un_region",
        "un_sub_region",
        "year",
        "inflation_pct"
    ]].sort_values(["un_region", "country_name_wb", "year"])

    return merged


final_df = merge_datasets(un_regions_df, inflation_df)

print(f" Merged dataset shape: {final_df.shape}")
print(f" UN Regions covered: {final_df['un_region'].dropna().unique().tolist()}")
print()
final_df.head(15)

---
##  Step 5 — Quick Exploration

In [ ]:
# Summary statistics by UN Region
print(" Average Inflation by UN Region (all years):")
final_df.groupby("un_region")["inflation_pct"].agg(["mean", "median", "max", "min"]).round(2)

In [ ]:
# Top 10 highest inflation instances
print(" Top 10 Highest Inflation Records:")
final_df.nlargest(10, "inflation_pct")[["country_name_wb", "un_region", "year", "inflation_pct"]]

---
##  Step 6 — Export CSV

In [ ]:
output_file = "inflation_by_un_region.csv"
final_df.to_csv(output_file, index=False)

print(f" CSV saved as '{output_file}'")
print(f"   Rows    : {len(final_df):,}")
print(f"   Columns : {list(final_df.columns)}")

In [ ]:
from google.colab import files
files.download(output_file)